# Uncovering Turbulent Dynamics in Stenotic Flows from 4D-flow MRI Measurements via Resolvent Analysis and Data Assimilation

**Paper:** Villie, A., Dillinger, H., Schmitter, S., Demange, S., Oberleithner, K. (2026). *Uncovering Turbulent Dynamics in Stenotic Flows from 4D-flow MRI Measurements via Resolvent Analysis and Data Assimilation.* arXiv:2606.03838 [physics.flu-dyn].

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Uncovering Turbulent Dynamics in Stenotic Flows from 4D-flow MRI.pdf`

## Como se usan las PINNs en este paper

El paper usa una **PINN de asimilacion de datos en dos pasos** para reconstruir el flujo medio turbulento en una estenosis (arteria estrechada) a partir de mediciones de resonancia magnetica 4D-flow, ruidosas y con un **artefacto de desplazamiento** (offset espacial por retraso de codificacion entre componentes de velocidad):

**Paso 1 &mdash; Correccion del artefacto:** una PINN $\mathbf{x}\mapsto\tilde{\mathbf{y}}(\mathbf{x})$ asimila un campo de velocidad medio libre de divergencia, descartando los puntos de entrenamiento donde $\bar u_x>0.65\max(\bar u_x)$ (la region mas afectada por el artefacto), con perdida fisica (Eq. 1):

$$\mathcal{L}_{PDE}=\|\nabla\cdot\bar{\mathbf{u}}(\mathbf{x}_c)\| + |\dot m(\mathbf{x}_m)-U_t\pi d_t^2/4|$$

es decir, continuidad en el interior **mas** una restriccion de caudal masico constante en varias secciones axiales. Condiciones de contorno: no-deslizamiento en la pared, axisimetria en el eje.

**Paso 2 &mdash; Asimilacion de presion y viscosidad turbulenta:** una segunda PINN, usando el campo de velocidad ya corregido como dato fijo, predice $[\tilde p,\sqrt{\nu_t}]$ (la raiz cuadrada garantiza $\nu_t\geq0$) minimizando el residuo de las ecuaciones RANS con cierre de viscosidad turbulenta (Eq. 2):

$$[\mathcal{L}_{mx},\mathcal{L}_{mr}]^T = \Big\|(\bar{\mathbf{u}}\cdot\nabla)\bar{\mathbf{u}}+\nabla\tilde p-\frac{1}{Re}\big(1+\frac{\nu_t}{\nu_m}\big)\Delta\bar{\mathbf{u}}\Big\|$$

Ambas redes son totalmente conectadas (4 capas x 128 neuronas en el paper), entrenadas con L-BFGS. El campo medio asimilado sirve luego de estado base para analisis de estabilidad lineal y analisis resolvente (no reproducidos aqui, al no ser PINNs).

Este cuaderno reproduce fielmente las **dos etapas de la PINN de asimilacion** sobre una geometria de estenosis axisimetrica sintetica con contraccion tipo coseno (75% de reduccion de area, igual que el fantoma del paper), incluyendo el artefacto de desplazamiento sintetico y su correccion.

## Repositorio publico de referencia

El PDF no incluye un repositorio propio (aunque cita el trabajo previo del mismo grupo, Villie et al. 2025, *Physics of Fluids*, que desarrolla el mismo framework de dos pasos, tampoco con repo publico conocido). Como referencia publica del mismo tipo de problema (PINNs para flujo sanguineo en estenosis usando datos dispersos), el paper cita explicitamente:

- **amir-cardiolab/PINN-wss** &mdash; https://github.com/amir-cardiolab/PINN-wss — codigo oficial de Arzani et al. (2021), "Uncovering near-wall blood flow from sparse data with physics-informed neural networks", con un ejemplo especifico de estenosis 2D.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Geometria de la estenosis (contraccion coseno, 75% de reduccion de area) y datos sinteticos con artefacto

In [ ]:
R0, Rt = 1.0, 0.5   # radio no obstruido y radio de garganta (75% reduccion de area: (Rt/R0)^2=0.25)
x_start, x_throat, x_end = 3.0, 5.5, 8.0
x_min, x_max = 0.0, 12.0
Q = np.pi * R0**2   # caudal (Ut=1 en la garganta tras normalizar)

def R_wall(x):
    x = np.atleast_1d(x)
    R = np.full_like(x, R0, dtype=float)
    m1 = (x >= x_start) & (x < x_throat)
    R[m1] = R0 - (R0 - Rt) * 0.5 * (1 - np.cos(np.pi * (x[m1] - x_start) / (x_throat - x_start)))
    m2 = (x >= x_throat) & (x < x_end)
    R[m2] = Rt + (R0 - Rt) * 0.5 * (1 - np.cos(np.pi * (x[m2] - x_throat) / (x_end - x_throat)))
    return R

def true_ux(x, r):
    """Perfil parabolico que conserva el caudal Q en cada seccion (campo sintetico de referencia)."""
    Rx = R_wall(x)
    Ubar = Q / (np.pi * Rx**2)
    return 2 * Ubar * np.maximum(1 - (r / Rx)**2, 0)

def displacement_artifact(x):
    """Desplazamiento sintetico localizado cerca de la garganta (analogo al artefacto de codificacion)."""
    return 1.2 * np.exp(-((x - x_throat) / 1.0)**2)

# Datos 'medidos' (con artefacto): ux desplazado en x cerca de la garganta
n_data = 3000
x_d = np.random.uniform(x_min, x_max, n_data)
r_d = np.random.uniform(0, 1, n_data) * R_wall(x_d)
ux_measured = true_ux(x_d - displacement_artifact(x_d), r_d)

# Filtrado: se descartan puntos donde ux > 0.65*max(ux), zona mas afectada por el artefacto
threshold = 0.65 * ux_measured.max()
keep = ux_measured <= threshold
x_train, r_train, ux_train = x_d[keep], r_d[keep], ux_measured[keep]
print(f'{keep.sum()} de {n_data} puntos retenidos tras el filtrado del artefacto')

## 2. Paso 1: PINN de correccion del artefacto (campo de velocidad libre de divergencia, Eq. 1)

In [ ]:
class VelocityPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 2)]  # ux, ur
        self.net = nn.Sequential(*layers)

    def forward(self, xr):
        out = self.net(xr)
        return out[:, 0:1], out[:, 1:2]


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]


xr_train = torch.tensor(np.stack([x_train, r_train], axis=1), dtype=torch.float32, device=device)
ux_train_t = torch.tensor(ux_train, dtype=torch.float32, device=device).view(-1, 1)

N_col = 3000
x_col_np = np.random.uniform(x_min, x_max, N_col)
r_col_np = np.random.uniform(0.02, 0.98, N_col) * R_wall(x_col_np)
xr_col = torch.tensor(np.stack([x_col_np, r_col_np], axis=1), dtype=torch.float32,
                       device=device, requires_grad=True)

x_wall_np = np.linspace(x_min, x_max, 300)
xr_wall = torch.tensor(np.stack([x_wall_np, R_wall(x_wall_np)], axis=1),
                        dtype=torch.float32, device=device)
x_axis_np = np.linspace(x_min, x_max, 300)
xr_axis = torch.tensor(np.stack([x_axis_np, np.zeros_like(x_axis_np)], axis=1),
                        dtype=torch.float32, device=device, requires_grad=True)


def continuity_residual(model, xr):
    ux, ur = model(xr)
    ux_x = d_d(ux, xr, 0)
    ur_r = d_d(ur, xr, 1)
    r = xr[:, 1:2].clamp(min=1e-3)
    return ux_x + ur / r + ur_r


def step1_loss(model):
    ux_pred, _ = model(xr_train)
    loss_data = torch.mean((ux_pred - ux_train_t)**2)

    div = continuity_residual(model, xr_col)
    loss_div = torch.mean(div**2)

    ux_wall, ur_wall = model(xr_wall)
    loss_wall = torch.mean(ux_wall**2 + ur_wall**2)  # no-deslizamiento

    _, ur_axis = model(xr_axis)
    loss_axis = torch.mean(ur_axis**2)  # axisimetria

    return loss_data + loss_div + 10.0 * (loss_wall + loss_axis)

In [ ]:
model1 = VelocityPINN().to(device)
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3)
hist1 = []
for epoch in range(3000):
    opt1.zero_grad()
    loss = step1_loss(model1)
    loss.backward()
    opt1.step()
    hist1.append(loss.item())
    if epoch % 500 == 0:
        print(f'[Paso 1] epoch {epoch:5d} | loss={loss.item():.4e}')

## 3. Paso 2: PINN de presion y viscosidad turbulenta (RANS con cierre de viscosidad de remolino, Eq. 2)

In [ ]:
Re = 3960.0  # numero de Reynolds del paper (basado en el diametro de garganta)

class PressureViscosityPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 2)]  # p_tilde, sqrt(nu_t)
        self.net = nn.Sequential(*layers)

    def forward(self, xr):
        out = self.net(xr)
        p = out[:, 0:1]
        sqrt_nut = out[:, 1:2]
        return p, sqrt_nut**2   # elevar al cuadrado garantiza nu_t >= 0


def rans_residual(model1, model2, xr):
    ux, ur = model1(xr)
    p, nu_t = model2(xr)

    ux_x = d_d(ux, xr, 0); ux_r = d_d(ux, xr, 1)
    ur_x = d_d(ur, xr, 0); ur_r_ = d_d(ur, xr, 1)
    p_x = d_d(p, xr, 0); p_r = d_d(p, xr, 1)
    ux_xx = d_d(ux_x, xr, 0); ux_rr = d_d(ux_r, xr, 1)
    ur_xx = d_d(ur_x, xr, 0); ur_rr = d_d(ur_r_, xr, 1)

    diff_coef = (1.0 / Re) * (1.0 + nu_t)
    res_x = ux * ux_x + ur * ux_r + p_x - diff_coef * (ux_xx + ux_rr)
    res_r = ux * ur_x + ur * ur_r_ + p_r - diff_coef * (ur_xx + ur_rr)
    return res_x, res_r


def step2_loss(model1, model2):
    res_x, res_r = rans_residual(model1, model2, xr_col)
    loss_mom = torch.mean(res_x**2) + torch.mean(res_r**2)

    _, nut_wall = model2(xr_wall)
    loss_wall = torch.mean(nut_wall**2)  # nu_t = 0 en la pared

    p_axis, _ = model2(xr_axis)
    p_axis_r = d_d(p_axis, xr_axis, 1)
    loss_axis = torch.mean(p_axis_r**2)  # dp/dr = 0 en el eje

    return loss_mom + 10.0 * (loss_wall + loss_axis)

In [ ]:
for p in model1.parameters():
    p.requires_grad_(False)  # el campo de velocidad del Paso 1 queda fijo (Seccion 'Step 2' del paper)

model2 = PressureViscosityPINN().to(device)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
hist2 = []
for epoch in range(2000):
    opt2.zero_grad()
    loss = step2_loss(model1, model2)
    loss.backward()
    opt2.step()
    hist2.append(loss.item())
    if epoch % 500 == 0:
        print(f'[Paso 2] epoch {epoch:5d} | loss={loss.item():.4e}')

## 4. Resultados: campos asimilados (cf. estilo Fig. 3 del paper)

In [ ]:
n_side = 100
xs = np.linspace(x_min, x_max, n_side)
rs = np.linspace(0, R0, n_side)
Xg, Rg = np.meshgrid(xs, rs)
mask_solid = Rg > R_wall(Xg.ravel()).reshape(Xg.shape)

xr_grid = torch.tensor(np.stack([Xg.ravel(), Rg.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    ux_g, ur_g = model1(xr_grid)
    p_g, nut_g = model2(xr_grid)

fields = {'ux (corregido)': ux_g, 'ur (corregido)': ur_g, 'p_tilde': p_g, 'nu_t': nut_g}
fig, axes = plt.subplots(4, 1, figsize=(10, 10))
for ax, (name, field) in zip(axes, fields.items()):
    F = field.cpu().numpy().reshape(Xg.shape)
    F = np.ma.array(F, mask=mask_solid)
    im = ax.contourf(Xg, Rg, F, levels=30, cmap='RdBu_r')
    ax.plot(xs, R_wall(xs), 'k-', linewidth=1.5)
    ax.set_ylabel('r')
    ax.set_title(name)
    plt.colorbar(im, ax=ax, fraction=0.03)
axes[-1].set_xlabel('x')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(hist1); axes[0].set_title('Convergencia Paso 1 (velocidad)')
axes[1].semilogy(hist2); axes[1].set_title('Convergencia Paso 2 (presion y viscosidad)')
for ax in axes:
    ax.set_xlabel('Epoca'); ax.set_ylabel('Loss'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

El Paso 1 debe recuperar un campo axial suave y libre de divergencia pese al artefacto localizado cerca de la garganta ($x\approx5.5$), mientras que el Paso 2 revela una viscosidad turbulenta $\nu_t$ concentrada en la capa de cizalla separada aguas abajo de la garganta -- cualitativamente consistente con la Fig. 3(d) del paper. El campo medio asimilado resultante serviria como estado base para el analisis de estabilidad lineal global y el analisis resolvente que el paper desarrolla a continuacion (no reproducidos aqui, al no ser PINNs).